# Member 1 - Skill Extraction (v2)
Run top to bottom on a **GPU** runtime (Runtime > Change runtime type > T4). Contract: `extract_skills(job_text) -> list[str]`; richer output via `extract_detailed`.

Cells: 1 install, 2 ESCO upload, 3-5 write modules, 6 build extractor, 7 demo, 8 export mocks, 9 evaluate (Experiment A).

In [ ]:
!pip install -q transformers accelerate
# torch ships with Colab. flashtext is no longer needed (replaced by a built-in trie).

In [ ]:
from pathlib import Path
from google.colab import files
if not Path('skills_en.csv').exists():
    print('Upload ESCO v1.2.1 skills_en.csv (English classification CSV)')
    files.upload()
assert Path('skills_en.csv').exists(), 'skills_en.csv missing'
!mkdir -p src eval
!touch src/__init__.py eval/__init__.py

In [ ]:
%%writefile src/extraction.py
"""Member 1 - Skill extraction (v2).

Public contract (frozen with the team)
--------------------------------------
    extract_skills(job_text: str) -> list[str]

Extras for downstream members (all optional to use)
---------------------------------------------------
    SkillExtractor.extract_detailed(job_text) -> list[Skill]
        Skill has: text, count, sources, confidence, requirement,
        importance_hint (0..1, a seed for Member 2's `importance d`),
        esco_candidates (hint for Member 2's linker), spans (char offsets).
    SkillExtractor.extract_many(texts) -> list[list[Skill]]   (batched, fast)

Pipeline
--------
    clean -> segment (lines / bullets / sentences, section headings tracked)
          -> taggers: JobBERT skill model + JobBERT knowledge model (batched)
          -> dictionaries: ESCO phrase trie + curated tech lexicon
          -> refine (trim fillers, split "X and <verb> Y", drop junk)
          -> consolidate overlapping spans by character offsets
          -> requirement level (required / preferred / ...) + importance hint

Design notes
------------
* Character offsets always refer to the string the caller passed in.
* torch / transformers are imported lazily, so everything except the HF
  taggers can be unit-tested without them (see tests/).
* A tagger is any object with `.tag(list[str]) -> list[list[(start, end, prob)]]`.
"""
from __future__ import annotations

import copy
import csv
import json
import re
import sys
import warnings
from dataclasses import dataclass, field
from typing import Dict, Iterable, List, Optional, Sequence, Set, Tuple

ESCO_VERSION = "1.2.1"
SKILL_MODEL = "jjzha/jobbert_skill_extraction"
KNOWLEDGE_MODEL = "jjzha/jobbert_knowledge_extraction"

SRC_SKILL = "jobbert_skill"
SRC_KNOWLEDGE = "jobbert_knowledge"
SRC_ESCO = "esco"
SRC_TECH = "tech"

# Prior reliability of the non-model sources (model sources use their softmax prob).
SOURCE_PRIOR = {SRC_ESCO: 0.75, SRC_TECH: 0.95}

# --------------------------------------------------------------------------
# Tech lexicon: canonical name -> aliases. Matched on whole tokens.
# Names in CASE_SENSITIVE only match with the exact casing written here
# (avoids "excel in teamwork", "pandas" the animal, "swift response", ...).
# --------------------------------------------------------------------------
TECH_LEXICON: Dict[str, List[str]] = {
    # languages
    "Python": ["Python"], "SQL": ["SQL"], "JavaScript": ["JavaScript", "JS"],
    "TypeScript": ["TypeScript"], "Java": ["Java"], "PHP": ["PHP"],
    "C++": ["C++"], "C#": ["C#"], "Golang": ["Golang"], "Rust": ["Rust"],
    "Kotlin": ["Kotlin"], "Swift": ["Swift"], "Ruby": ["Ruby"], "Scala": ["Scala"],
    "Dart": ["Dart"], "Julia": ["Julia"], "R": ["R programming", "R language", "RStudio"],
    "Bash": ["Bash", "shell scripting"], "NoSQL": ["NoSQL"],
    "HTML": ["HTML"], "HTML5": ["HTML5", "HTML 5"], "CSS": ["CSS"], "CSS3": ["CSS3", "CSS 3"],
    # data / analytics
    "Excel": ["Excel", "EXCEL", "MS Excel", "Microsoft Excel"],
    "Google Sheets": ["Google Sheets"],
    "Pandas": ["Pandas"], "NumPy": ["NumPy"], "SciPy": ["SciPy"],
    "scikit-learn": ["scikit-learn", "sklearn"],
    "Matplotlib": ["Matplotlib"], "Seaborn": ["Seaborn"], "Plotly": ["Plotly"],
    "Power BI": ["Power BI", "PowerBI"], "Tableau": ["Tableau"],
    "Looker": ["Looker"], "Looker Studio": ["Looker Studio", "Data Studio"],
    "Power Query": ["Power Query"], "Power Automate": ["Power Automate"],
    "DAX": ["DAX"], "Star Schema": ["Star Schema"], "ETL": ["ETL", "ELT"],
    "data warehousing": ["data warehousing", "data warehouse"],
    "data modeling": ["data modeling", "data modelling"],
    "data transformation": ["data transformation"],
    "data cleansing": ["data cleansing", "data cleaning", "cleansing"],
    "statistics": ["statistics"], "A/B testing": ["A/B testing", "A/B tests", "A/B test"],
    "Jupyter": ["Jupyter", "Jupyter Notebook"], "Streamlit": ["Streamlit"],
    "Spark": ["Spark", "Apache Spark"], "PySpark": ["PySpark"], "Hadoop": ["Hadoop"],
    "Airflow": ["Airflow", "Apache Airflow"], "dbt": ["dbt"], "Kafka": ["Kafka"],
    "Snowflake": ["Snowflake"], "BigQuery": ["BigQuery"], "Hive": ["Hive"],
    "SSIS": ["SSIS"], "Salesforce": ["Salesforce"], "SAP": ["SAP"],
    # databases
    "PostgreSQL": ["PostgreSQL", "Postgres"], "MySQL": ["MySQL"], "SQLite": ["SQLite"],
    "SQL Server": ["SQL Server", "MSSQL"], "Oracle": ["Oracle"],
    "MongoDB": ["MongoDB"], "Redis": ["Redis"], "Elasticsearch": ["Elasticsearch"],
    "Firebase": ["Firebase"], "Supabase": ["Supabase"],
    # web / backend
    "React.js": ["React.js", "ReactJS", "React"], "React Native": ["React Native"],
    "Next.js": ["Next.js", "NextJS"], "Node.js": ["Node.js", "NodeJS"],
    "Express.js": ["Express.js", "ExpressJS"], "Vue.js": ["Vue.js", "VueJS", "Vue"],
    "Angular": ["Angular"], "Svelte": ["Svelte"], "Redux": ["Redux"],
    "Tailwind CSS": ["Tailwind CSS", "Tailwind"], "Bootstrap": ["Bootstrap"],
    "Django": ["Django"], "Flask": ["Flask"], "FastAPI": ["FastAPI"],
    "Spring Boot": ["Spring Boot"], ".NET": [".NET"], "GraphQL": ["GraphQL"],
    "REST APIs": ["REST APIs", "REST API", "RESTful APIs", "RESTful"],
    "WordPress": ["WordPress"], "Shopify": ["Shopify"], "Flutter": ["Flutter"],
    # devops / cloud / tools
    "Git": ["Git"], "GitHub": ["GitHub"], "GitLab": ["GitLab"], "Bitbucket": ["Bitbucket"],
    "Docker": ["Docker"], "Kubernetes": ["Kubernetes", "K8s"], "Terraform": ["Terraform"],
    "Jenkins": ["Jenkins"], "Linux": ["Linux"], "Nginx": ["Nginx"],
    "AWS": ["AWS", "Amazon Web Services"],
    "Azure": ["Azure", "Microsoft Azure"],
    "GCP": ["GCP", "Google Cloud Platform", "Google Cloud"],
    "CI/CD pipelines": ["CI/CD pipelines", "CI/CD pipeline", "CI/CD"],
    "automated testing": ["automated testing", "test automation"],
    "Selenium": ["Selenium"], "Cypress": ["Cypress"], "Jest": ["Jest"],
    "Pytest": ["Pytest"], "Postman": ["Postman"],
    "Jira": ["Jira"], "Confluence": ["Confluence"], "ClickUp": ["ClickUp"],
    "Notion": ["Notion"], "Slack": ["Slack"], "Figma": ["Figma"],
    "Photoshop": ["Photoshop"],
    "Scrum": ["Scrum"], "Agile": ["Agile"], "Kanban": ["Kanban"],
    "sprint planning": ["sprint planning"],
    # ML / AI
    "TensorFlow": ["TensorFlow"], "PyTorch": ["PyTorch"], "Keras": ["Keras"],
    "XGBoost": ["XGBoost"], "ONNX": ["ONNX"], "TensorRT": ["TensorRT"],
    "Triton Inference Server": ["Triton Inference Server"],
    "Hugging Face Transformers": ["Hugging Face Transformers", "Hugging Face"],
    "LangChain": ["LangChain"], "LlamaIndex": ["LlamaIndex"],
    "Pinecone": ["Pinecone"], "Weaviate": ["Weaviate"], "pgvector": ["pgvector"],
    "embeddings": ["embeddings"], "vector search": ["vector search"],
    "vector databases": ["vector databases", "vector database"],
    "machine learning": ["machine learning", "ML"],
    "deep learning": ["deep learning"],
    "NLP": ["NLP", "natural language processing"],
    "computer vision": ["computer vision"],
    "prompt engineering": ["prompt engineering"],
    "LLMs": ["LLMs", "LLM", "large language models"],
    "RAG": ["RAG", "retrieval-augmented generation"],
    "GenAI": ["GenAI", "generative AI"], "OpenAI API": ["OpenAI API"],
    # marketing
    "Google Analytics": ["Google Analytics"], "Google Ads": ["Google Ads", "AdWords"],
    "Google Tag Manager": ["Google Tag Manager"], "Meta Ads": ["Meta Ads", "Facebook Ads"],
    "SEO": ["SEO"], "SEM": ["SEM"], "keyword research": ["keyword research"],
    "on-page optimization": ["on-page optimization"], "off-page SEO": ["off-page SEO"],
    "affiliate marketing": ["affiliate marketing"],
    "email marketing": ["email marketing"], "Mailchimp": ["Mailchimp"],
    "HubSpot": ["HubSpot"], "Ahrefs": ["Ahrefs"], "SEMrush": ["SEMrush"],
}

CASE_SENSITIVE: Set[str] = {
    "Excel", "Pandas", "Git", "Swift", "Rust", "Ruby", "Scala", "Dart", "Julia",
    "Flask", "Hive", "Spark", "Looker", "Oracle", "Angular", "Notion", "Slack",
    "Jest", "Cypress", "Bootstrap", "Redux", "Vue.js", "React.js", "Snowflake",
    "Kafka", "Airflow", "ML", "RAG", "SEM", "JS", "R",
}
# canonical -> tokens that, when they directly follow the match, cancel it.
NEG_NEXT: Dict[str, Set[str]] = {
    "Excel": {"in", "at", "as", "under", "from", "beyond", "with"},
    "Spark": {"joy", "creativity", "innovation"},
}
# Aliases that are only valid when the exact casing is used even if the
# canonical is not in CASE_SENSITIVE (acronyms colliding with common words).
CASE_SENSITIVE_ALIASES: Set[str] = {"JS", "ML", "RAG", "SEM", "R", "Vue", "React", "Hive", "SAP", "DAX"}

ACTION_VERBS = set(
    "analyze analyse build create develop design manage perform preprocess implement "
    "maintain monitor optimize optimise coordinate communicate evaluate test write train "
    "deploy assess present prepare document review lead conduct collaborate clean "
    "visualize visualise automate integrate query extract report".split()
)
_STOP = set(
    "a an the and or of to in on for with at by from as is are be been being this that "
    "these those it its you we our your their they he she i not no yes can will may "
    "must should would could also more most other such any all each both etc".split()
)
_GENERIC = {
    "experience", "skills", "skill", "ability", "abilities", "team", "teams", "work",
    "working", "years", "year", "knowledge", "role", "position", "candidate", "company",
    "job", "opportunity", "environment", "responsibilities", "requirements", "plus",
    "degree", "bachelor", "master", "background", "understanding", "someone", "person",
    "people", "other", "etc", "equal opportunity employer", "benefits", "salary",
    "time", "day", "days", "month", "months", "week", "weeks", "bonus", "preferred",
}
_LEAD_FILLERS = re.compile(
    r"^(?:(?:the|a|an|and|or|with|in|of|to|for|on|as|strong|good|excellent|solid|proven)\s+)+",
    re.I,
)
_EDGE_PUNCT = " \t\r\n,;:.!?()[]{}\"'`*-–—•"


def alias_to_canonical() -> Dict[str, str]:
    """casefolded alias -> casefolded canonical, e.g. 'react' -> 'react.js' (for alias-aware scoring)."""
    out: Dict[str, str] = {}
    for canonical, aliases in TECH_LEXICON.items():
        for a in aliases + [canonical]:
            out[a.casefold()] = canonical.casefold()
    return out


# --------------------------------------------------------------------------
# Config & data classes
# --------------------------------------------------------------------------
@dataclass
class ExtractorConfig:
    use_skill_model: bool = True
    use_knowledge_model: bool = True
    use_esco: bool = True
    use_tech: bool = True
    min_model_prob: float = 0.0     # drop model spans below this mean token prob
    max_span_words: int = 8         # longer spans are almost always sentences, not skills
    max_segment_words: int = 70     # hard cap per tagger input (keeps well under 512 tokens)
    esco_min_words: int = 2         # single-word ESCO labels are too noisy; tech lexicon covers tools
    batch_size: int = 16
    split_compounds: bool = True


@dataclass
class Segment:
    start: int
    end: int
    text: str
    section: str  # required | preferred | responsibility | company | unknown


@dataclass
class Mention:
    start: int
    end: int
    text: str
    src_conf: Dict[str, float]
    label: Optional[str] = None            # display override: canonical tech name, or expanded verb phrase
    esco: Set[str] = field(default_factory=set)
    requirement: str = "unknown"

    @property
    def sources(self) -> Set[str]:
        return set(self.src_conf)

    @property
    def confidence(self) -> float:
        miss = 1.0
        for c in self.src_conf.values():
            miss *= 1.0 - max(0.0, min(1.0, c))
        return round(1.0 - miss, 4)

    @property
    def display(self) -> str:
        return self.label or self.text

    def merge(self, other: "Mention") -> None:
        for s, c in other.src_conf.items():
            self.src_conf[s] = max(self.src_conf.get(s, 0.0), c)
        self.esco |= other.esco
        if other.label and not self.label and (other.start, other.end) == (self.start, self.end):
            self.label = other.label


IMPORTANCE = {"required": 1.0, "responsibility": 0.8, "unknown": 0.7, "preferred": 0.5, "company": 0.3}


@dataclass
class Skill:
    text: str
    count: int
    sources: List[str]
    confidence: float
    requirement: str
    importance_hint: float
    esco_candidates: List[str]
    spans: List[Tuple[int, int]]

    def to_dict(self) -> dict:
        return {
            "skill": self.text, "count": self.count, "sources": self.sources,
            "confidence": self.confidence, "requirement": self.requirement,
            "importance_hint": self.importance_hint,
            "esco_candidates": self.esco_candidates, "spans": [list(s) for s in self.spans],
        }


# --------------------------------------------------------------------------
# Text utilities
# --------------------------------------------------------------------------
_TOKEN_RE = re.compile(r"\w+|[^\w\s]")
_NBSP_MAP = {ord(" "): " ", ord("​"): " ", ord(" "): " ", ord(" "): " ",
             ord("\r"): " ", ord("\t"): " ", ord("’"): "'", ord("‘"): "'",
             ord("“"): '"', ord("”"): '"'}


def clean_text(text: str) -> str:
    """Length-preserving cleanup, so offsets stay valid for the caller's string."""
    return text.translate(_NBSP_MAP)


def tokenize(text: str) -> List[Tuple[str, int, int]]:
    return [(m.group(), m.start(), m.end()) for m in _TOKEN_RE.finditer(text)]


def normalize(text: str) -> str:
    return " ".join(text.split())


class PhraseTrie:
    """Token trie, case-insensitive with optional exact-case entries, longest match."""

    def __init__(self) -> None:
        self.root: dict = {}
        self.size = 0

    def add(self, tokens: Sequence[str], payload, case_sensitive: bool = False) -> None:
        node = self.root
        for t in tokens:
            node = node.setdefault(t.casefold(), {})
        node.setdefault(None, []).append((payload, tuple(tokens) if case_sensitive else None))
        self.size += 1

    def find_all(self, toks: List[Tuple[str, int, int]]):
        """Yield (first_token_idx, last_token_idx_exclusive, [payloads])."""
        out, i, n = [], 0, len(toks)
        while i < n:
            node, best, j = self.root, None, i
            while j < n:
                nxt = node.get(toks[j][0].casefold())
                if nxt is None:
                    break
                node, j = nxt, j + 1
                if None in node:
                    valid = [p for p, cs in node[None]
                             if cs is None or all(toks[i + k][0] == cs[k] for k in range(len(cs)))]
                    if valid:
                        best = (j, valid)
            if best:
                out.append((i, best[0], best[1]))
                i = best[0]
            else:
                i += 1
        return out


# --------------------------------------------------------------------------
# Segmentation (lines -> bullets -> sentences) with section tracking
# --------------------------------------------------------------------------
_BULLET_RE = re.compile(r"^\s*(?:[-*•▪●◦‣·–—]|\(?\d{1,2}[.)])\s+")
_SENT_SPLIT = re.compile(r"(?<=[.!?;])\s+(?=[A-Z0-9(\"'])")
_INLINE_BULLET = re.compile(r"\s+[•▪●◦‣·]\s+")
_ABBREV_END = ("e.g.", "i.e.", "vs.", "approx.", "etc.")
_COLON_RE = re.compile(r"^([^:\n]{2,70}?):\s*(.*)$")

_SECTION_PATTERNS = [
    ("preferred", r"nice[- ]to[- ]have|preferred|bonus|desirable|good to have|a plus|plus points?|optional|advantageous|beneficial"),
    ("company", r"about (?:us|the (?:company|team))|who we are|benefits|perks|what we offer|we offer|why (?:join|work)|our (?:mission|culture|values)|compensation|salary|equal opportunity"),
    ("responsibility", r"responsibilit|what you(?:'ll| will) do|duties|your role|the role|day[- ]to[- ]day|job description|key tasks|you will"),
    ("required", r"requirements?|qualifications?|must[- ]haves?|what you(?:'ll)? (?:bring|need)|you have|skills?|who you are|about you|minimum|essential|required|looking for|experience|tech(?:nical)? stack"),
]
_EXACT_HEADINGS = {
    "requirements": "required", "qualifications": "required", "minimum qualifications": "required",
    "required qualifications": "required", "required skills": "required", "must have": "required",
    "must haves": "required", "what you bring": "required", "what you need": "required",
    "who you are": "required", "about you": "required", "key skills": "required",
    "skills": "required", "technical skills": "required", "tech stack": "required",
    "skills and qualifications": "required", "skills & qualifications": "required",
    "preferred qualifications": "preferred", "preferred skills": "preferred",
    "nice to have": "preferred", "nice to haves": "preferred", "bonus": "preferred",
    "bonus points": "preferred", "good to have": "preferred", "desirable": "preferred",
    "responsibilities": "responsibility", "key responsibilities": "responsibility",
    "duties": "responsibility", "what you will do": "responsibility", "what you'll do": "responsibility",
    "your role": "responsibility", "the role": "responsibility", "job description": "responsibility",
    "about us": "company", "about the company": "company", "who we are": "company",
    "benefits": "company", "perks": "company", "what we offer": "company",
    "why join us": "company", "compensation": "company",
}
_PREF_CUES = re.compile(
    r"\b(?:nice[- ]to[- ]have|(?:is|are|would be|will be)\s+(?:a\s+|an\s+)?(?:plus|bonus|advantage|nice|beneficial|desirable|preferred|great|welcome)"
    r"|a plus|bonus points?|preferred|desirable|good to have|ideally|advantageous|an advantage|not required|optional)\b", re.I)
_REQ_CUES = re.compile(
    r"\b(?:must|required|requires?|essential|mandatory|needs? to|proficien\w+|expert\w*|proven|strong|solid|at least \d+|\d+\+?\s*years?)\b", re.I)


def _norm_heading(s: str) -> str:
    s = s.casefold().replace("’", "'")
    return " ".join(re.sub(r"[^a-z&' ]+", " ", s).split())


def _classify_colon_heading(h: str) -> Optional[str]:
    if len(h.split()) > 8:
        return None
    low = h.casefold()
    for sec, pat in _SECTION_PATTERNS:
        if re.search(pat, low):
            return sec
    return None


def _heading(body: str) -> Optional[Tuple[str, str, int]]:
    """Return (section, rest_text, rest_offset) when the line is/starts with a heading."""
    m = _COLON_RE.match(body)
    if m:
        sec = _classify_colon_heading(m.group(1))
        if sec:
            return sec, m.group(2), m.start(2)
    if len(body.split()) <= 5:
        sec = _EXACT_HEADINGS.get(_norm_heading(body))
        if sec:
            return sec, "", len(body)
    return None


def _strip_range(text: str, a: int, b: int) -> Tuple[int, int]:
    while a < b and text[a].isspace():
        a += 1
    while b > a and text[b - 1].isspace():
        b -= 1
    return a, b


def _split_by(text: str, a: int, b: int, pattern: "re.Pattern") -> List[Tuple[int, int]]:
    out, pos = [], a
    for m in pattern.finditer(text, a, b):
        if m.end() > m.start():
            out.append(_strip_range(text, pos, m.start()))
            pos = m.end()
    out.append(_strip_range(text, pos, b))
    return [(s, e) for s, e in out if e > s]


def _merge_abbrev(text: str, pieces: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    out: List[Tuple[int, int]] = []
    carry: Optional[Tuple[int, int]] = None
    for s, e in pieces:
        if carry:
            s = carry[0]
            carry = None
        if text[s:e].casefold().endswith(_ABBREV_END):
            carry = (s, e)
            continue
        out.append((s, e))
    if carry:
        out.append(carry)
    return out


def _wrap_words(text: str, a: int, b: int, max_words: int) -> List[Tuple[int, int]]:
    words = [(m.start() + a, m.end() + a) for m in re.finditer(r"\S+", text[a:b])]
    if len(words) <= max_words:
        return [(a, b)]
    out, i = [], 0
    while i < len(words):
        j = min(i + max_words, len(words))
        if j < len(words):  # prefer breaking after a comma in the last third of the window
            for k in range(j, i + (2 * max_words) // 3, -1):
                if text[words[k - 1][1] - 1] == ",":
                    j = k
                    break
        out.append((words[i][0], words[j - 1][1]))
        i = j
    return out


def segment(text: str, max_words: int = 70) -> List[Segment]:
    """Split into tagger-sized segments. text[seg.start:seg.end] == seg.text always."""
    segs: List[Segment] = []
    section = "unknown"
    for m in re.finditer(r"[^\n]+", text):
        a, b = _strip_range(text, m.start(), m.end())
        if a >= b:
            continue
        bm = _BULLET_RE.match(text[a:b])   # slice: '^' would anchor at string start, not at `a`
        if bm:
            a, b = _strip_range(text, a + bm.end(), b)
            if a >= b:
                continue
        local = section
        h = _heading(text[a:b])
        if h:
            sec, rest, rest_off = h
            if not rest.strip():
                section = sec
                continue
            local = sec
            a, b = _strip_range(text, a + rest_off, b)
            if a >= b:
                continue
        pieces: List[Tuple[int, int]] = []
        for s, e in _split_by(text, a, b, _INLINE_BULLET):
            pieces.extend(_split_by(text, s, e, _SENT_SPLIT))
        for s, e in _merge_abbrev(text, pieces):
            for s2, e2 in _wrap_words(text, s, e, max_words):
                segs.append(Segment(s2, e2, text[s2:e2], local))
    return segs


def requirement_for(seg: Segment) -> str:
    if seg.section == "company":
        return "company"
    if _PREF_CUES.search(seg.text):
        return "preferred"
    if _REQ_CUES.search(seg.text):
        return "required"
    return seg.section if seg.section in IMPORTANCE else "unknown"


# --------------------------------------------------------------------------
# HuggingFace token-classification tagger (lazy torch import)
# --------------------------------------------------------------------------
class HFTokenTagger:
    """Batched B/I/O span tagger. Works with the SkillSpan JobBERT checkpoints."""

    def __init__(self, model_name: str, batch_size: int = 16, max_length: int = 256, device: Optional[str] = None):
        import torch
        from transformers import AutoModelForTokenClassification, AutoTokenizer

        self.torch = torch
        self.model_name = model_name
        self.batch_size = batch_size
        self.max_length = max_length
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        if not self.tokenizer.is_fast:
            raise RuntimeError("A fast tokenizer is required for character offsets.")
        self.model = AutoModelForTokenClassification.from_pretrained(model_name).to(self.device).eval()
        id2label = self.model.config.id2label
        self._letters = {int(i): str(l).strip().upper()[:1] for i, l in id2label.items()}
        if not set(self._letters.values()) <= {"B", "I", "O"} or "O" not in self._letters.values():
            raise RuntimeError(f"Unexpected label set for {model_name}: {id2label}")

    def tag(self, texts: List[str]) -> List[List[Tuple[int, int, float]]]:
        torch = self.torch
        results: List[List[Tuple[int, int, float]]] = [[] for _ in texts]
        order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
        for lo in range(0, len(order), self.batch_size):
            idx = order[lo:lo + self.batch_size]
            batch = [texts[i] for i in idx]
            enc = self.tokenizer(batch, return_offsets_mapping=True, truncation=True,
                                 max_length=self.max_length, padding=True, return_tensors="pt")
            offsets = enc.pop("offset_mapping").tolist()
            with torch.inference_mode():
                logits = self.model(**{k: v.to(self.device) for k, v in enc.items()}).logits
                probs = torch.softmax(logits, dim=-1)
                conf, lab = probs.max(dim=-1)
            conf, lab = conf.cpu().tolist(), lab.cpu().tolist()
            for row, i in enumerate(idx):
                results[i] = self._decode(lab[row], conf[row], enc.word_ids(row), offsets[row])
        return results

    def _decode(self, labels, confs, word_ids, offsets) -> List[Tuple[int, int, float]]:
        spans: List[Tuple[int, int, float]] = []
        cur: Optional[list] = None
        prev_word = None

        def flush():
            if cur is not None:
                spans.append((cur[0], cur[1], sum(cur[2]) / len(cur[2])))

        for k, wid in enumerate(word_ids):
            if wid is None:
                continue
            s, e = offsets[k]
            if wid == prev_word:                    # sub-word piece: extend, never relabel
                if cur is not None:
                    cur[1] = max(cur[1], e)
                continue
            prev_word = wid
            tag = self._letters[labels[k]]          # label of the first sub-word decides
            if tag == "O":
                flush()
                cur = None
            elif tag == "B" or cur is None:
                flush()
                cur = [s, e, [confs[k]]]
            else:                                   # "I"
                cur[1] = e
                cur[2].append(confs[k])
        flush()
        return spans


# --------------------------------------------------------------------------
# Span refinement helpers
# --------------------------------------------------------------------------
def _trim(text: str, a: int, b: int) -> Tuple[int, int]:
    while a < b and text[a] in _EDGE_PUNCT:
        a += 1
    while b > a and text[b - 1] in _EDGE_PUNCT:
        b -= 1
    m = _LEAD_FILLERS.match(text[a:b])  # slice: see note in segment()
    if m and a + m.end() < b:
        a += m.end()
    tail = re.search(r"\s+(?:and|or)$", text[a:b], re.I)
    if tail:
        b = a + tail.start()
    while b > a and text[b - 1] in _EDGE_PUNCT:
        b -= 1
    return a, b


def _valid_span(span: str, max_words: int) -> bool:
    if len(span) < 2 or not re.search(r"[A-Za-z]", span):
        return False
    words = span.split()
    if len(words) > max_words:
        return False
    low = span.casefold()
    if low in _GENERIC:
        return False
    toks = [t.casefold() for t in re.findall(r"\w+", span)]
    if not toks or all(t in _STOP or t.isdigit() for t in toks):
        return False
    if all(t in _STOP or t in _GENERIC or t.isdigit() for t in toks):
        return False
    return True


def _split_compound(text: str, a: int, b: int) -> List[Tuple[int, int]]:
    m = list(re.finditer(r"\s+and\s+", text[a:b], re.I))
    if len(m) == 1:
        right = text[a + m[0].end():b].split()
        if right and right[0].casefold() in ACTION_VERBS:
            return [(a, a + m[0].start()), (a + m[0].end(), b)]
    return [(a, b)]



_SEP_RE = re.compile(r"\s*(?:,\s*)?(?:(?:and|or|&|/)\s*)?", re.I)


def _fix_verb_fragments(text: str, ms: List[Mention]) -> List[Mention]:
    """Repair verb coordination the tagger splits, e.g. 'build and maintain web apps'.

    The model tags 'build' and 'maintain web apps' separately. A lone action verb is
    never a useful skill, so: if it is followed (through 'and' / ',' / 'or') by a
    verb-led phrase, give it that phrase's object ('build web apps'); otherwise drop it.
    The expanded wording goes in `label`, while the span still points at the verb.
    """
    model_only = {SRC_SKILL, SRC_KNOWLEDGE}
    ms = sorted(ms, key=lambda m: m.start)
    out: List[Mention] = []
    pending: List[Mention] = []
    prev_end = 0

    def is_lone_verb(m: Mention) -> bool:
        return (len(m.text.split()) == 1 and m.text.casefold() in ACTION_VERBS
                and set(m.src_conf) <= model_only)

    for m in ms:
        gap_ok = bool(pending) and _SEP_RE.fullmatch(text[prev_end:m.start]) is not None
        if is_lone_verb(m):
            if pending and not gap_ok:
                pending = []                      # earlier verbs had nothing to attach to: dropped
            pending.append(m)
        else:
            words = m.text.split(None, 1)
            if gap_ok and len(words) == 2 and words[0].casefold() in ACTION_VERBS:
                for v in pending:
                    v.label = f"{v.text} {words[1]}"
                    out.append(v)
            pending = []
            out.append(m)
        prev_end = m.end
    return out

# --------------------------------------------------------------------------
# The extractor
# --------------------------------------------------------------------------
class SkillExtractor:
    def __init__(
        self,
        esco_csv: Optional[str] = None,
        config: Optional[ExtractorConfig] = None,
        skill_tagger=None,
        knowledge_tagger=None,
        extra_tech: Optional[Dict[str, List[str]]] = None,
        load_models: bool = True,
    ):
        self.cfg = config or ExtractorConfig()
        self._cache: Dict[Tuple[str, str], List[Tuple[int, int, float]]] = {}
        self.skill_tagger = skill_tagger
        self.knowledge_tagger = knowledge_tagger
        if load_models:
            if self.skill_tagger is None and self.cfg.use_skill_model:
                self.skill_tagger = HFTokenTagger(SKILL_MODEL, self.cfg.batch_size)
            if self.knowledge_tagger is None and self.cfg.use_knowledge_model:
                try:
                    self.knowledge_tagger = HFTokenTagger(KNOWLEDGE_MODEL, self.cfg.batch_size)
                except Exception as exc:  # optional second model must never break the pipeline
                    warnings.warn(f"Knowledge model unavailable ({exc!r}); continuing without it.")
                    self.knowledge_tagger = None
        self.esco_trie = PhraseTrie()
        self.esco_lookup: Dict[Tuple[str, ...], Set[str]] = {}
        self.esco_records = 0
        if esco_csv:
            self._load_esco(esco_csv)
        self._lexicon = dict(TECH_LEXICON)
        for k, v in (extra_tech or {}).items():
            self._lexicon[k] = list(dict.fromkeys(self._lexicon.get(k, []) + list(v)))
        self.tech_trie = self._build_tech_trie(self._lexicon)

    @staticmethod
    def _build_tech_trie(lexicon: Dict[str, List[str]], exclude: Iterable[str] = ()) -> PhraseTrie:
        trie, skip = PhraseTrie(), set(exclude)
        for canonical, aliases in lexicon.items():
            if canonical in skip:
                continue
            for alias in aliases:
                toks = [t for t, _, _ in tokenize(alias)]
                cs = canonical in CASE_SENSITIVE or alias in CASE_SENSITIVE_ALIASES
                trie.add(toks, canonical, case_sensitive=cs)
        return trie

    def without_tech_terms(self, canonicals: Iterable[str]) -> "SkillExtractor":
        """Clone (shared models/cache) whose tech lexicon lacks the given canonical names.
        Used to bound how much of an eval score comes from lexicon entries written while looking at the eval ads."""
        clone = copy.copy(self)
        clone.tech_trie = self._build_tech_trie(self._lexicon, canonicals)
        return clone

    # ---- construction helpers ------------------------------------------
    def with_config(self, **overrides) -> "SkillExtractor":
        """Cheap copy that shares models, tries and the tagger cache (used for ablations)."""
        clone = copy.copy(self)
        clone.cfg = ExtractorConfig(**{**self.cfg.__dict__, **overrides})
        return clone

    def _load_esco(self, path: str) -> None:
        with open(path, newline="", encoding="utf-8-sig") as fh:
            reader = csv.DictReader(fh)
            cols = set(reader.fieldnames or [])
            missing = {"preferredLabel", "altLabels"} - cols
            if missing:
                raise ValueError(f"ESCO skills CSV is missing columns: {sorted(missing)}")
            label_cols = [c for c in ("preferredLabel", "altLabels", "hiddenLabels") if c in cols]
            for row in reader:
                preferred = normalize(row.get("preferredLabel", ""))
                if not preferred:
                    continue
                self.esco_records += 1
                phrases = [preferred]
                for c in label_cols[1:]:
                    phrases += (row.get(c) or "").splitlines()
                for phrase in phrases:
                    toks = tuple(t.casefold() for t, _, _ in tokenize(normalize(phrase)))
                    if len(toks) < self.cfg.esco_min_words:
                        continue
                    self.esco_lookup.setdefault(toks, set()).add(preferred)
        for toks, labels in self.esco_lookup.items():
            self.esco_trie.add(toks, labels)

    # ---- tagging -------------------------------------------------------
    def _model_sources(self):
        if self.cfg.use_skill_model and self.skill_tagger is not None:
            yield SRC_SKILL, self.skill_tagger
        if self.cfg.use_knowledge_model and self.knowledge_tagger is not None:
            yield SRC_KNOWLEDGE, self.knowledge_tagger

    def _ensure_tagged(self, seg_texts: Iterable[str]) -> None:
        uniq = list(dict.fromkeys(seg_texts))
        for name, tagger in self._model_sources():
            todo = [t for t in uniq if (name, t) not in self._cache]
            if todo:
                for t, spans in zip(todo, tagger.tag(todo)):
                    self._cache[(name, t)] = spans

    def clear_cache(self) -> None:
        self._cache.clear()

    # ---- per-text pipeline --------------------------------------------
    def _refine_model_span(self, text, a, b, prob, source) -> List[Mention]:
        a, b = _trim(text, a, b)
        ranges = _split_compound(text, a, b) if self.cfg.split_compounds else [(a, b)]
        out = []
        for s, e in ranges:
            s, e = _trim(text, s, e)
            if e > s and _valid_span(text[s:e], self.cfg.max_span_words):
                out.append(Mention(s, e, text[s:e], {source: prob}))
        return out

    def _segment_mentions(self, text: str, seg: Segment) -> List[Mention]:
        model_ms: List[Mention] = []
        for name, _ in self._model_sources():
            for s, e, p in self._cache.get((name, seg.text), []):
                if p >= self.cfg.min_model_prob:
                    model_ms += self._refine_model_span(text, seg.start + s, seg.start + e, p, name)
        ms: List[Mention] = _fix_verb_fragments(text, model_ms) if self.cfg.split_compounds else model_ms
        toks = None
        if self.cfg.use_esco and self.esco_trie.size:
            toks = tokenize(seg.text)
            for i, j, payloads in self.esco_trie.find_all(toks):
                a, b = seg.start + toks[i][1], seg.start + toks[j - 1][2]
                labels = set().union(*payloads)
                ms.append(Mention(a, b, text[a:b], {SRC_ESCO: SOURCE_PRIOR[SRC_ESCO]}, esco=labels))
        if self.cfg.use_tech:
            toks = toks or tokenize(seg.text)
            for i, j, payloads in self.tech_trie.find_all(toks):
                canonical = payloads[0]
                if j < len(toks) and toks[j][0].casefold() in NEG_NEXT.get(canonical, ()):
                    continue
                a, b = seg.start + toks[i][1], seg.start + toks[j - 1][2]
                ms.append(Mention(a, b, text[a:b], {SRC_TECH: SOURCE_PRIOR[SRC_TECH]}, label=canonical))
        return _consolidate(ms)

    def _mentions(self, text: str) -> List[Mention]:
        text = clean_text(text)
        segs = segment(text, self.cfg.max_segment_words)
        self._ensure_tagged(s.text for s in segs)
        out: List[Mention] = []
        for seg in segs:
            req = requirement_for(seg)
            for m in self._segment_mentions(text, seg):
                m.requirement = req
                out.append(m)
        return out

    def _to_skills(self, mentions: List[Mention]) -> List[Skill]:
        groups: Dict[str, List[Mention]] = {}
        for m in sorted(mentions, key=lambda m: (m.start, -(m.end - m.start))):
            groups.setdefault(m.display.casefold(), []).append(m)
        skills = []
        for ms in groups.values():
            best = max(ms, key=lambda m: (IMPORTANCE[m.requirement], m.confidence))
            src: Dict[str, float] = {}
            labels: Set[str] = set()
            for m in ms:
                for s, c in m.src_conf.items():
                    src[s] = max(src.get(s, 0.0), c)
                labels |= m.esco
            key = tuple(t.casefold() for t, _, _ in tokenize(best.display))
            labels |= self.esco_lookup.get(key, set())
            miss = 1.0
            for c in src.values():
                miss *= 1.0 - c
            skills.append(Skill(
                text=best.display, count=len(ms), sources=sorted(src),
                confidence=round(1.0 - miss, 4), requirement=best.requirement,
                importance_hint=IMPORTANCE[best.requirement],
                esco_candidates=sorted(labels)[:5], spans=[(m.start, m.end) for m in ms],
            ))
        return skills

    # ---- public API ----------------------------------------------------
    def extract_detailed(self, job_text: str, min_confidence: float = 0.0) -> List[Skill]:
        if not isinstance(job_text, str):
            raise TypeError("job_text must be a string")
        if not job_text.strip():
            return []
        skills = self._to_skills(self._mentions(job_text))
        return [s for s in skills if s.confidence >= min_confidence]

    def extract_skills(self, job_text: str) -> List[str]:
        """Frozen contract: str -> list[str] (unique, document order)."""
        return [s.text for s in self.extract_detailed(job_text)]

    def extract_many(self, texts: Sequence[str]) -> List[List[Skill]]:
        """Same as extract_detailed per text, but tags all segments in one batched pass."""
        cleaned = []
        for t in texts:
            if not isinstance(t, str):
                raise TypeError("every job text must be a string")
            cleaned.append(clean_text(t))
        self._ensure_tagged(s.text for t in cleaned for s in segment(t, self.cfg.max_segment_words))
        return [self.extract_detailed(t) for t in texts]

    def extract_skills_many(self, texts: Sequence[str]) -> List[List[str]]:
        return [[s.text for s in skills] for skills in self.extract_many(texts)]


def _consolidate(ms: List[Mention]) -> List[Mention]:
    """Resolve overlaps by character offsets (never by string containment)."""
    by_range: Dict[Tuple[int, int], Mention] = {}
    for m in ms:
        key = (m.start, m.end)
        if key in by_range:
            by_range[key].merge(m)
        else:
            by_range[key] = m
    keep: List[Mention] = []
    for m in sorted(by_range.values(), key=lambda m: (m.start, -(m.end - m.start))):
        placed = False
        for idx, k in enumerate(keep):
            overlap = min(k.end, m.end) - max(k.start, m.start)
            if overlap <= 0:
                continue
            m_in_k = k.start <= m.start and m.end <= k.end
            if m_in_k:
                if SRC_TECH in m.src_conf:      # a tool inside a longer phrase stays linkable
                    continue
                k.merge(m)
                placed = True
                break
            shorter = min(k.end - k.start, m.end - m.start)
            if overlap / shorter >= 0.5:
                if (m.end - m.start) > (k.end - k.start) and SRC_TECH not in k.src_conf:
                    m.merge(k)
                    keep[idx] = m
                else:
                    k.merge(m)
                placed = True
                break
        if not placed:
            keep.append(m)
    return sorted(keep, key=lambda m: m.start)


# --------------------------------------------------------------------------
# Module-level handoff function (what other members import)
# --------------------------------------------------------------------------
_DEFAULT: Optional[SkillExtractor] = None


def get_default_extractor(esco_csv: str = "skills_en.csv") -> SkillExtractor:
    global _DEFAULT
    if _DEFAULT is None:
        import os
        path = os.environ.get("ESCO_SKILLS_CSV", esco_csv)
        if not os.path.exists(path):
            warnings.warn(f"ESCO CSV not found at {path!r}; running without the ESCO dictionary.")
            path = None
        _DEFAULT = SkillExtractor(path)
    return _DEFAULT


def extract_skills(job_text: str) -> List[str]:
    """Public Member 1 interface: str -> list[str]."""
    return get_default_extractor().extract_skills(job_text)


def to_handoff(skills: List[Skill]) -> List[dict]:
    """Rich JSON for Member 2 / Member 5 (superset of the frozen list[str] contract)."""
    return [s.to_dict() for s in skills]


if __name__ == "__main__":  # python -m src.extraction "job text..."  |  < ad.txt
    ad = " ".join(sys.argv[1:]) or sys.stdin.read()
    print(json.dumps(to_handoff(get_default_extractor().extract_detailed(ad)), indent=2, ensure_ascii=False))


In [ ]:
%%writefile eval/evaluate.py
"""Experiment A harness for Member 1.

Gold format (JSONL, one ad per line):
    {"id": "ad01", "text": "...job ad...", "skills": ["write SQL queries", "Python", ...]}

Usage:
    python -m eval.evaluate --gold eval/sample_gold.jsonl --esco skills_en.csv
    python -m eval.evaluate --gold my_50_ads.jsonl --esco skills_en.csv --no-models   # dictionary-only baseline

Reports micro precision / recall / F1 under two matching rules:
  exact    - normalized strings are equal
  partial  - exact first, then greedy one-to-one match on token Jaccard >= 0.5
and an ablation over the extractor's sources, plus the most common false
positives / false negatives (that list is the fastest way to improve the system).
"""
from __future__ import annotations

import argparse
import json
import re
import sys
from collections import Counter
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

try:
    from src.extraction import ExtractorConfig, SkillExtractor, alias_to_canonical
except ImportError:  # running inside the Colab notebook, where the module sits next to us
    from extraction import ExtractorConfig, SkillExtractor, alias_to_canonical


def _base_norm(s: str) -> str:
    s = s.casefold().replace("’", "'")
    s = re.sub(r"[^\w+#./ -]+", " ", s)
    return " ".join(s.split()).strip(" .-/")


_ALIAS = {_base_norm(k): _base_norm(v) for k, v in alias_to_canonical().items()}


def norm(s: str) -> str:
    """Normalize for matching. Known tool aliases collapse ('React' == 'React.js', 'A/B tests' == 'A/B testing')."""
    n = _base_norm(s)
    return _ALIAS.get(n, n)


def _jaccard(a: str, b: str) -> float:
    ta, tb = set(a.split()), set(b.split())
    return len(ta & tb) / len(ta | tb) if ta and tb else 0.0


def match_counts(pred: Sequence[str], gold: Sequence[str], partial: bool, thr: float = 0.5) -> Tuple[int, List[str], List[str]]:
    """Return (true_positives, unmatched_pred, unmatched_gold) with one-to-one matching."""
    pred_n = list(dict.fromkeys(norm(p) for p in pred if norm(p)))
    gold_n = list(dict.fromkeys(norm(g) for g in gold if norm(g)))
    gold_left, pred_left = list(gold_n), []
    tp = 0
    for p in pred_n:
        if p in gold_left:
            gold_left.remove(p)
            tp += 1
        else:
            pred_left.append(p)
    if partial:
        still = []
        for p in pred_left:
            best, best_j = None, 0.0
            for g in gold_left:
                j = _jaccard(p, g)
                if j > best_j:
                    best, best_j = g, j
            if best is not None and best_j >= thr:
                gold_left.remove(best)
                tp += 1
            else:
                still.append(p)
        pred_left = still
    return tp, pred_left, gold_left


def prf(tp: int, n_pred: int, n_gold: int) -> Dict[str, float]:
    p = tp / n_pred if n_pred else 0.0
    r = tp / n_gold if n_gold else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return {"precision": round(p, 4), "recall": round(r, 4), "f1": round(f, 4)}


def score(preds: List[List[str]], golds: List[List[str]]):
    out, fps, fns = {}, Counter(), Counter()
    for mode in ("exact", "partial"):
        tp = n_pred = n_gold = 0
        for p, g in zip(preds, golds):
            t, p_left, g_left = match_counts(p, g, partial=(mode == "partial"))
            tp += t
            n_pred += len(set(norm(x) for x in p if norm(x)))
            n_gold += len(set(norm(x) for x in g if norm(x)))
            if mode == "partial":
                fps.update(p_left)
                fns.update(g_left)
        out[mode] = prf(tp, n_pred, n_gold)
    return out, fps, fns


def load_gold(path: str) -> List[dict]:
    rows = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if line.strip():
            row = json.loads(line)
            if not isinstance(row.get("text"), str) or not isinstance(row.get("skills"), list):
                raise ValueError(f"Bad gold row (need 'text' str and 'skills' list): {line[:80]}")
            rows.append(row)
    return rows


ABLATIONS = {
    "skill model only": dict(use_skill_model=True, use_knowledge_model=False, use_esco=False, use_tech=False),
    "+ knowledge model": dict(use_skill_model=True, use_knowledge_model=True, use_esco=False, use_tech=False),
    "+ ESCO dictionary": dict(use_skill_model=True, use_knowledge_model=True, use_esco=True, use_tech=False),
    "+ tech lexicon (full)": dict(use_skill_model=True, use_knowledge_model=True, use_esco=True, use_tech=True),
    "dictionaries only": dict(use_skill_model=False, use_knowledge_model=False, use_esco=True, use_tech=True),
}


def run_ablation(extractor: SkillExtractor, rows: List[dict]):
    texts = [r["text"] for r in rows]
    golds = [r["skills"] for r in rows]
    table, details = [], {}
    for name, flags in ABLATIONS.items():
        ex = extractor.with_config(**flags)
        needs_skill = flags["use_skill_model"] and ex.skill_tagger is None
        knowledge_only_row = name == "+ knowledge model" and ex.knowledge_tagger is None
        if needs_skill or knowledge_only_row:   # model not loaded: row would be meaningless
            continue
        preds = ex.extract_skills_many(texts)
        res, fps, fns = score(preds, golds)
        table.append((name, res))
        details[name] = (preds, fps, fns)
    # Lower bound: remove every lexicon entry whose name/alias is itself a gold phrase. The lexicon was
    # partly written while looking at the dev ads, so the full row is optimistic and this row is pessimistic
    # (it also removes legitimate entries such as Python). The truth for unseen ads lies between them.
    gold_base = {_base_norm(g) for gs in golds for g in gs}
    exclude = {c for c, al in extractor._lexicon.items() if any(_base_norm(a) in gold_base for a in al + [c])}
    full = extractor.with_config(**ABLATIONS["+ tech lexicon (full)"])
    if full.skill_tagger is not None:
        ex = full.without_tech_terms(exclude)
        preds = ex.extract_skills_many(texts)
        res, fps, fns = score(preds, golds)
        name = f"lexicon minus gold terms ({len(exclude)} removed)"
        table.append((name, res))
        details[name] = (preds, fps, fns)
    return table, details


def by_category(extractor: SkillExtractor, rows: List[dict]) -> Dict[str, dict]:
    """Partial-match P/R/F1 per row['category'] for the full configuration."""
    out = {}
    for cat in sorted({r.get("category", "all") for r in rows}):
        sub = [r for r in rows if r.get("category", "all") == cat]
        preds = extractor.extract_skills_many([r["text"] for r in sub])
        out[cat] = score(preds, [r["skills"] for r in sub])[0]["partial"]
    return out


def format_table(table) -> str:
    head = f"{'configuration':<26}| {'P(exact)':>8} {'R(exact)':>8} {'F1(exact)':>9} | {'P(part)':>7} {'R(part)':>7} {'F1(part)':>8}"
    lines = [head, "-" * len(head)]
    for name, r in table:
        e, p = r["exact"], r["partial"]
        lines.append(f"{name:<26}| {e['precision']:>8.3f} {e['recall']:>8.3f} {e['f1']:>9.3f} | {p['precision']:>7.3f} {p['recall']:>7.3f} {p['f1']:>8.3f}")
    return "\n".join(lines)


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--gold", required=True)
    ap.add_argument("--esco", default=None, help="path to ESCO skills_en.csv")
    ap.add_argument("--no-models", action="store_true", help="skip loading JobBERT (dictionary-only)")
    ap.add_argument("--out", default="eval/results.json")
    ap.add_argument("--top", type=int, default=15, help="how many FP/FN strings to print")
    args = ap.parse_args(argv)

    rows = load_gold(args.gold)
    print(f"{len(rows)} ads, {sum(len(r['skills']) for r in rows)} gold skills")
    extractor = SkillExtractor(args.esco, load_models=not args.no_models)
    table, details = run_ablation(extractor, rows)
    print("\n" + format_table(table))
    print("\nNOTE: the 'full' row is optimistic if the tech lexicon was written while looking at these ads;"
          "\n      'lexicon minus gold terms' is the pessimistic bound. Unseen-ad performance lies between them."
          "\n      Rows without the tech lexicon (model / +knowledge / +ESCO) are unaffected by this.")
    full_name = "+ tech lexicon (full)" if "+ tech lexicon (full)" in details else table[-1][0]
    _, fps, fns = details[full_name]
    print(f"\nTop false positives ({full_name}):")
    for s, c in fps.most_common(args.top):
        print(f"  {c:>2}x  {s}")
    print(f"\nTop false negatives ({full_name}):")
    for s, c in fns.most_common(args.top):
        print(f"  {c:>2}x  {s}")
    if any("category" in r for r in rows) and not args.no_models:
        print("\nPartial-match P/R/F1 by category (full pipeline):")
        for cat, m in by_category(extractor, rows).items():
            print(f"  {cat:<24} P={m['precision']:.2f} R={m['recall']:.2f} F1={m['f1']:.2f}")
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    Path(args.out).write_text(json.dumps({n: r for n, r in table}, indent=2), encoding="utf-8")
    print(f"\nSaved {args.out}")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile eval/prelabel.py
"""Speed up hand-labeling: run the extractor over raw ads and write a draft gold file.

Input : a .jsonl with {"id","text"} per line, OR a .txt where ads are separated by a line of '====='.
Output: .jsonl in gold format, "skills" pre-filled with the model's guesses.
Then open the output, DELETE wrong skills and ADD missed ones. Correcting is ~3x faster than labeling from scratch.
Note: pre-labeling biases the gold toward the model; add every missed skill you can see, and label blind
(--blank) for a sample of ~10 ads to measure that bias.

    python -m eval.prelabel ads.txt eval/gold_draft.jsonl --esco skills_en.csv
"""
from __future__ import annotations

import argparse
import json
import re
from pathlib import Path

try:
    from src.extraction import SkillExtractor
except ImportError:
    from extraction import SkillExtractor


def read_ads(path: str):
    raw = Path(path).read_text(encoding="utf-8")
    if path.endswith(".jsonl"):
        for i, line in enumerate(l for l in raw.splitlines() if l.strip()):
            row = json.loads(line)
            yield row.get("id", f"ad{i+1:03d}"), row["text"]
    else:
        for i, chunk in enumerate(c.strip() for c in re.split(r"^\s*={3,}\s*$", raw, flags=re.M) if c.strip()):
            yield f"ad{i+1:03d}", chunk


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("ads")
    ap.add_argument("out")
    ap.add_argument("--esco", default=None)
    ap.add_argument("--blank", action="store_true", help="write empty skill lists (blind labeling)")
    args = ap.parse_args(argv)
    ads = list(read_ads(args.ads))
    ex = None if args.blank else SkillExtractor(args.esco)
    preds = [[] for _ in ads] if args.blank else ex.extract_skills_many([t for _, t in ads])
    with open(args.out, "w", encoding="utf-8") as fh:
        for (ad_id, text), skills in zip(ads, preds):
            fh.write(json.dumps({"id": ad_id, "text": text, "skills": skills}, ensure_ascii=False) + "\n")
    print(f"Wrote {len(ads)} ads to {args.out}")


if __name__ == "__main__":
    main()



In [ ]:
%%writefile eval/convert_annotations.py
"""Convert (jobs.csv + offset annotations.csv) -> gold JSONL for eval.evaluate.

    python -m eval.convert_annotations data/jobs_linkedin_50.csv data/annotations_draft_50.csv data/gold.jsonl

Validates that every annotation's character offsets really select its phrase in the job text
(blank offsets are allowed for expanded phrases and skip that check),
and reports problems instead of silently scoring against misaligned labels.
Re-run it after the human review pass; only rows you keep in the CSV reach the gold file.
"""
from __future__ import annotations

import csv
import json
import sys
from collections import defaultdict


def convert(jobs_csv: str, ann_csv: str, out_jsonl: str, text_col: str = "job_description") -> dict:
    jobs = {r["job_id"]: r for r in csv.DictReader(open(jobs_csv, encoding="utf-8-sig", newline=""))}
    by, problems, statuses = defaultdict(list), [], set()
    for a in csv.DictReader(open(ann_csv, encoding="utf-8-sig", newline="")):
        job = jobs.get(a["job_id"])
        if job is None:
            problems.append(f"{a['job_id']}: annotation for unknown job")
            continue
        text = job[text_col]
        if str(a.get("start_char", "")).strip() or str(a.get("end_char", "")).strip():
            s, e = int(a["start_char"]), int(a["end_char"])
            if text[s:e] != a["skill_phrase"]:
                problems.append(f"{a['job_id']}: offsets {s}:{e} select {text[s:e]!r}, not {a['skill_phrase']!r}")
                continue
        # blank offsets = an expanded phrase that is not a contiguous substring ("collect data" from
        # "Collect, clean, and organize data"); accepted as-is, so leave start_char/end_char empty for those.
        statuses.add(a.get("annotation_status", ""))
        if a["skill_phrase"] not in by[a["job_id"]]:
            by[a["job_id"]].append(a["skill_phrase"])
    with open(out_jsonl, "w", encoding="utf-8") as f:
        for job_id, r in jobs.items():
            f.write(json.dumps({"id": job_id, "category": r.get("job_category", ""), "text": r[text_col],
                                "skills": by.get(job_id, [])}, ensure_ascii=False) + "\n")
    return {"ads": len(jobs), "annotations": sum(map(len, by.values())), "problems": problems, "statuses": sorted(statuses)}


if __name__ == "__main__":
    info = convert(*sys.argv[1:4])
    print(info)
    sys.exit(1 if info["problems"] else 0)



In [ ]:
import sys, time
sys.path.insert(0, '.')
import torch
from src.extraction import SkillExtractor, ExtractorConfig, to_handoff
t0 = time.time()
extractor = SkillExtractor('skills_en.csv')
print('device       :', extractor.skill_tagger.device)
print('knowledge mdl:', 'on' if extractor.knowledge_tagger else 'OFF (see warning above)')
print('ESCO records :', extractor.esco_records, '| ESCO phrases:', len(extractor.esco_lookup))
print(f'load time    : {time.time()-t0:.0f}s')

In [ ]:
import pandas as pd
example = ('We need a Data Analyst proficient in Python, SQL and Power BI. '
           'Build dashboards, analyze customer data and perform statistical analysis. '
           'Experience with TensorFlow, LangChain and Next.js is a plus.')
skills = extractor.extract_detailed(example)
pd.DataFrame([s.to_dict() for s in skills]).drop(columns=['spans'])

In [ ]:
import json, pathlib
from src.extraction import to_handoff
pathlib.Path('mock').mkdir(exist_ok=True)
json.dump([s.text for s in skills], open('mock/extracted_skills.json','w'), indent=2, ensure_ascii=False)
json.dump(to_handoff(skills), open('mock/extracted_skills_detailed.json','w'), indent=2, ensure_ascii=False)
print(open('mock/extracted_skills.json').read())

## Experiment A - evaluation
`sample_gold.jsonl` below is a **5-ad synthetic dev set** for smoke-testing only. For the real result, hand-label ~50 ads (use `eval/prelabel.py` to draft) and point `GOLD` at that file.

In [ ]:
%%writefile eval/sample_gold.jsonl
{"id": "dev01", "text": "Data Analyst\n\nAbout us:\nWe are a fast-growing fintech with great benefits and a friendly team.\n\nRequirements:\n- 2+ years of experience with Python and SQL\n- Build dashboards in Power BI or Tableau\n- Strong communication skills\n\nNice to have:\n- Experience with Docker\n- Knowledge of machine learning", "skills": ["Python", "SQL", "build dashboards", "Power BI", "Tableau", "communication skills", "Docker", "machine learning"]}
{"id": "dev02", "text": "We are hiring a full stack developer. You will build and maintain web apps using React.js, Node.js and MongoDB. Experience with Git and REST APIs is required. Familiarity with AWS is a plus.", "skills": ["build web apps", "maintain web apps", "React.js", "Node.js", "MongoDB", "Git", "REST APIs", "AWS"]}
{"id": "dev03", "text": "Digital Marketing Executive. Responsibilities: plan SEO campaigns, perform keyword research and manage Google Ads budgets. Requirements: hands-on Google Analytics, email marketing, excellent written English. Excel in a fast-paced environment.", "skills": ["plan SEO campaigns", "SEO", "keyword research", "manage Google Ads budgets", "Google Ads", "Google Analytics", "email marketing", "written English"]}
{"id": "dev04", "text": "ML Engineer\n\nWhat you'll do:\n* Train and deploy deep learning models with PyTorch\n* Optimize inference using ONNX and TensorRT\n* Collaborate with product managers\n\nBonus:\n* LangChain, vector databases, prompt engineering", "skills": ["train deep learning models", "deploy deep learning models", "deep learning", "PyTorch", "optimize inference", "ONNX", "TensorRT", "collaborate with product managers", "LangChain", "vector databases", "prompt engineering"]}
{"id": "dev05", "text": "Project Coordinator. Run sprint planning and daily Scrum meetings in Jira. Prepare status reports for stakeholders; coordinate with cross-functional teams. Must be proficient in Microsoft Excel. PMP certification is a plus.", "skills": ["sprint planning", "Scrum", "Jira", "prepare status reports", "coordinate with cross-functional teams", "Excel"]}


In [ ]:
from eval.evaluate import load_gold, run_ablation, format_table
GOLD = 'eval/sample_gold.jsonl'   # <- replace with your 50-ad file
rows = load_gold(GOLD)
table, details = run_ablation(extractor, rows)
print(format_table(table))
_, fps, fns = details['+ tech lexicon (full)']
print('\nTop false positives:', fps.most_common(12))
print('Top false negatives:', fns.most_common(12))